# 视频横向拼接器
每行输入一个视频路径，可选添加 annotation（最多 6 个）。

In [ ]:
import os
import re
import base64
import mimetypes
import tempfile
import subprocess
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

POSITIONS = ["top", "bottom", "top-left", "top-right", "bottom-left", "bottom-right"]
MAX_ANNOTATIONS = 6

POSITION_EXPR = {
    "top":          "x=(w-text_w)/2:y=30",
    "bottom":       "x=(w-text_w)/2:y=h-text_h-30",
    "top-left":     "x=30:y=30",
    "top-right":    "x=w-text_w-30:y=30",
    "bottom-left":  "x=30:y=h-text_h-30",
    "bottom-right": "x=w-text_w-30:y=h-text_h-30",
}

FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
    "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/dejavu/DejaVuSans.ttf",
]
FONT_PATH = next((f for f in FONT_CANDIDATES if os.path.isfile(f)), None)

SUPPORTED_EXTS = (".mp4",)

# drawtext 中需要转义的字符（ffmpeg text= 参数语义）
_DRAWTEXT_ESCAPE = str.maketrans({
    "\\": r"\\",
    ":": r"\:",
    "'": r"\'",
    "%": r"\%",
})


def validate_path(path: str):
    if not path:
        return None, "路径为空"
    path = os.path.realpath(os.path.expanduser(path.strip()))
    if not path.lower().endswith(SUPPORTED_EXTS):
        return None, f"仅支持 .mp4: {path}"
    if not os.path.isfile(path):
        return None, f"文件不存在: {path}"
    return path, ""


def escape_drawtext(text: str) -> str:
    return text.translate(_DRAWTEXT_ESCAPE)


def expand_annotation(text: str, video_path: str) -> str:
    """将 /frame_idx /name 转义命令展开为 drawtext text= 可用的片段。

    - /name      -> 静态替换为 basename(video_path)
    - /frame_idx -> ffmpeg 动态表达式 %{eif:n:d}（当前帧编号）
    """
    name_val = escape_drawtext(os.path.basename(video_path))
    frame_expr = "%{eif\\:n\\:d}"  # 在 text= 里用，冒号要 \: 转义

    tokens = re.split(r"(/frame_idx|/name)", text)
    out_parts = []
    for tok in tokens:
        if tok == "/name":
            out_parts.append(name_val)
        elif tok == "/frame_idx":
            out_parts.append(frame_expr)
        else:
            out_parts.append(escape_drawtext(tok))
    return "".join(out_parts)


def build_drawtext(expanded_text: str, position: str) -> str:
    font_opt = f":fontfile={FONT_PATH}" if FONT_PATH else ""
    return (
        f"drawtext=text='{expanded_text}'"
        f"{font_opt}"
        f":fontcolor=white:fontsize=28"
        f":box=1:boxcolor=black@0.5:boxborderw=8"
        f":{POSITION_EXPR[position]}"
    )


def video_html(path: str, max_height: int = 600) -> str:
    mime, _ = mimetypes.guess_type(path)
    mime = mime or "video/mp4"
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode("ascii")
    return (
        f'<video controls style="max-width:100%;max-height:{max_height}px;" '
        f'src="data:{mime};base64,{data}"></video>'
    )


def run_concat(video_paths_text, ann_count, dropdowns, texts):
    if not video_paths_text.strip():
        return None, "请输入至少一个视频路径", []

    paths = [p.strip() for p in video_paths_text.strip().split("\n") if p.strip()]
    validated = []
    for p in paths:
        norm_path, err = validate_path(p)
        if err:
            return None, f"路径校验失败: {err}", []
        validated.append(norm_path)

    if not validated:
        return None, "没有有效视频", []

    n = len(validated)

    per_video = [[] for _ in range(n)]
    for i in range(int(ann_count)):
        position = dropdowns[i] or "top"
        block = texts[i] or ""
        lines = block.split("\n")
        for vid_idx in range(n):
            line = lines[vid_idx].strip() if vid_idx < len(lines) else ""
            if line:
                per_video[vid_idx].append((position, line))

    try:
        filter_parts = []
        for vid_idx in range(n):
            chain = f"[{vid_idx}:v]"
            if per_video[vid_idx]:
                drawtexts = []
                for position, text in per_video[vid_idx]:
                    expanded = expand_annotation(text, validated[vid_idx])
                    drawtexts.append(build_drawtext(expanded, position))
                chain += ",".join(drawtexts)
            else:
                chain += "null"
            chain += f"[vann{vid_idx}]"
            filter_parts.append(chain)

        hstack_inputs = "".join(f"[vann{i}]" for i in range(n))
        filter_parts.append(f"{hstack_inputs}hstack=inputs={n}[v]")

        audio_inputs = "".join(f"[{i}:a]" for i in range(n))
        filter_parts.append(f"{audio_inputs}amerge=inputs={n}[a]")

        filter_complex = ";".join(filter_parts)

        output_path = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False, dir="/tmp").name

        cmd = ["ffmpeg", "-y"]
        for v in validated:
            cmd += ["-i", v]
        cmd += [
            "-filter_complex", filter_complex,
            "-map", "[v]", "-map", "[a]",
            "-c:v", "libx264", "-preset", "fast", "-crf", "23",
            "-c:a", "aac", "-b:a", "192k",
            output_path,
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            return None, f"FFmpeg 错误:\n{result.stderr[-2000:]}", []

        previews = []
        for vid_idx in range(n):
            previews.append((validated[vid_idx], per_video[vid_idx]))

        return output_path, f"成功拼接 {n} 个视频，应用 {int(ann_count)} 个 annotation", previews

    except Exception as e:
        return None, f"处理异常: {e}", []

In [ ]:
path_input = widgets.Textarea(
    description="视频路径",
    placeholder="每行一个，例如:\n/path/to/video1.mp4\n/path/to/video2.mp4",
    layout=widgets.Layout(width="100%", height="120px"),
    style={"description_width": "80px"},
)
concat_btn = widgets.Button(description="拼接", button_style="primary", icon="play")

ann_count_state = {"value": 0}
ann_rows = []
ann_dropdowns = []
ann_texts = []

for i in range(MAX_ANNOTATIONS):
    dd = widgets.Dropdown(
        options=POSITIONS, value=POSITIONS[i] if i < len(POSITIONS) else POSITIONS[0],
        description=f"#{i+1} 位置",
        layout=widgets.Layout(width="220px"),
        style={"description_width": "60px"},
    )
    tb = widgets.Textarea(
        description=f"#{i+1} 文本",
        placeholder=(
            "第1个视频的文本\n第2个视频的文本\n...\n"
            "支持转义：/frame_idx 当前帧编号，/name 视频文件名"
        ),
        layout=widgets.Layout(width="100%", height="70px"),
        style={"description_width": "60px"},
    )
    row = widgets.HBox(
        [dd, tb],
        layout=widgets.Layout(display="none", width="100%", margin="4px 0"),
    )
    ann_rows.append(row)
    ann_dropdowns.append(dd)
    ann_texts.append(tb)

add_btn = widgets.Button(description="+ 新增 annotation")
remove_btn = widgets.Button(description="- 删除最后一个")

_updating_options = {"flag": False}


def refresh_dropdown_options(_change=None):
    if _updating_options["flag"]:
        return
    _updating_options["flag"] = True
    try:
        count = ann_count_state["value"]
        active_vals = [ann_dropdowns[i].value for i in range(count)]
        for i in range(count):
            current = ann_dropdowns[i].value
            used_by_others = {v for j, v in enumerate(active_vals) if j != i and v is not None}
            allowed = [p for p in POSITIONS if p == current or p not in used_by_others]
            if list(ann_dropdowns[i].options) != allowed:
                ann_dropdowns[i].options = allowed
                ann_dropdowns[i].value = current
    finally:
        _updating_options["flag"] = False


for dd in ann_dropdowns:
    dd.observe(refresh_dropdown_options, names="value")


def refresh_rows():
    for i, row in enumerate(ann_rows):
        row.layout.display = "flex" if i < ann_count_state["value"] else "none"
    refresh_dropdown_options()


def _on_add(_):
    if ann_count_state["value"] >= MAX_ANNOTATIONS:
        return
    next_idx = ann_count_state["value"]
    used = {ann_dropdowns[i].value for i in range(next_idx)}
    default = next((p for p in POSITIONS if p not in used), POSITIONS[0])
    _updating_options["flag"] = True
    try:
        ann_dropdowns[next_idx].options = POSITIONS
        ann_dropdowns[next_idx].value = default
    finally:
        _updating_options["flag"] = False
    ann_count_state["value"] = next_idx + 1
    refresh_rows()


def _on_remove(_):
    ann_count_state["value"] = max(ann_count_state["value"] - 1, 0)
    refresh_rows()


add_btn.on_click(_on_add)
remove_btn.on_click(_on_remove)

status_out = widgets.Output()
result_out = widgets.Output()
previews_out = widgets.Output()


def on_concat(_):
    with status_out:
        clear_output()
        print("处理中…")
    with result_out:
        clear_output()
    with previews_out:
        clear_output()

    dd_vals = [d.value for d in ann_dropdowns]
    tx_vals = [t.value for t in ann_texts]
    out_path, msg, previews = run_concat(
        path_input.value, ann_count_state["value"], dd_vals, tx_vals,
    )

    with status_out:
        clear_output()
        color = "red" if out_path is None else "green"
        display(HTML(f'<div style="color:{color};white-space:pre-wrap;">{msg}</div>'))

    if out_path:
        with result_out:
            display(HTML("<h3 style='text-align:center;'>拼接结果</h3>"))
            mime, _ = mimetypes.guess_type(out_path)
            mime = mime or "video/mp4"
            with open(out_path, "rb") as f:
                data = base64.b64encode(f.read()).decode("ascii")
            display(HTML(
                f'<div style="display:flex;justify-content:center;width:100%;">'
                f'<video controls style="max-width:100%;max-height:600px;" '
                f'src="data:{mime};base64,{data}"></video>'
                f'</div>'
            ))

        with previews_out:
            display(HTML("<h3>各视频预览</h3>"))
            for idx, (vp, anns) in enumerate(previews):
                ann_html = "".join(
                    f"<li><b>[{pos}]</b> {txt}</li>" for pos, txt in anns
                ) or "<li><i>(无)</i></li>"
                display(HTML(
                    f'<div style="display:flex;gap:16px;align-items:flex-start;'
                    f'border-top:1px solid #ddd;padding:12px 0;">'
                    f'<div style="flex:0 0 320px;">'
                    f'<video controls style="width:320px;" src="data:video/mp4;base64,'
                    f'{base64.b64encode(open(vp, "rb").read()).decode("ascii")}"></video>'
                    f'</div>'
                    f'<div style="flex:1;">'
                    f'<div><b>视频 {idx+1}</b></div>'
                    f'<div style="word-break:break-all;"><b>路径:</b> {vp}</div>'
                    f'<div><b>Annotations:</b><ul>{ann_html}</ul></div>'
                    f'</div>'
                    f'</div>'
                ))


concat_btn.on_click(on_concat)

ui = widgets.VBox([
    widgets.HBox([path_input, concat_btn], layout=widgets.Layout(align_items="flex-start")),
    widgets.HTML(
        "<h3 style='margin-bottom:4px;'>Annotations</h3>"
        "<div style='color:#666;font-size:12px;margin-bottom:6px;'>"
        "文本中可使用转义：<code>/frame_idx</code> 显示当前帧编号（动态），"
        "<code>/name</code> 显示视频文件名。"
        "</div>"
    ),
    widgets.HBox([add_btn, remove_btn]),
    widgets.VBox(ann_rows),
    status_out,
    result_out,
    previews_out,
])

display(ui)